In [ ]:
!pip install -q pypdf python-docx sentence-transformers faiss-cpu gradio transformers sentencepiece

In [ ]:
from google.colab import files

uploaded_files = files.upload()

print("Total files uploaded:", len(uploaded_files))

Saving CV35.pdf to CV35.pdf
Saving CV36.pdf to CV36.pdf
Saving CV37.pdf to CV37.pdf
Saving CV795.pdf to CV795.pdf
Saving CV798.pdf to CV798.pdf
Saving CV1177.pdf to CV1177.pdf
Saving CV1263.pdf to CV1263.pdf
Saving CV1295.pdf to CV1295.pdf
Saving CV1917.doc to CV1917.doc
Saving CV3188.doc to CV3188.doc
Saving CV3693.pdf to CV3693.pdf
Saving CV4146.pdf to CV4146.pdf
Saving CV5687.pdf to CV5687.pdf
Saving CV5695.pdf to CV5695.pdf
Saving CV6217.doc to CV6217.doc
Saving CV6425.pdf to CV6425.pdf
Saving CV6568.pdf to CV6568.pdf
Saving CV7309.doc to CV7309.doc
Saving CV7889.pdf to CV7889.pdf
Saving CV8279.doc to CV8279.doc
Saving CV8299.pdf to CV8299.pdf
Saving CV8316.pdf to CV8316.pdf
Saving CV8381.docx to CV8381.docx
Saving CV8671.pdf to CV8671.pdf
Saving CV8875.doc to CV8875.doc
Saving CV9308.doc to CV9308.doc
Saving CV9458.pdf to CV9458.pdf
Saving CV9497.pdf to CV9497.pdf
Saving CV9504.pdf to CV9504.pdf
Saving CV9534.doc to CV9534.doc
Saving CV10077.pdf to CV10077.pdf
Saving CV10088.pdf t

In [ ]:
import os
from pypdf import PdfReader
from docx import Document

all_cvs = []

for filename in uploaded_files.keys():

    text = ""

    try:
        if filename.lower().endswith(".pdf"):

            reader = PdfReader(filename)

            for page in reader.pages:
                text += page.extract_text() or ""

        elif filename.lower().endswith(".docx"):

            document = Document(filename)

            for paragraph in document.paragraphs:
                text += paragraph.text + "\n"

        else:
            print("Unsupported file:", filename)
            continue

        if text.strip():

            all_cvs.append({
                "filename": filename,
                "text": text
            })

            print("Successfully extracted:", filename)

        else:
            print("No readable text:", filename)

    except Exception as e:
        print("Error reading", filename, ":", e)

print("\nTotal readable CVs:", len(all_cvs))

Successfully extracted: CV35.pdf
Successfully extracted: CV36.pdf
Successfully extracted: CV37.pdf
Successfully extracted: CV795.pdf
Successfully extracted: CV798.pdf
Successfully extracted: CV1177.pdf
Successfully extracted: CV1263.pdf
Successfully extracted: CV1295.pdf
Unsupported file: CV1917.doc
Unsupported file: CV3188.doc
Successfully extracted: CV3693.pdf
Successfully extracted: CV4146.pdf
Successfully extracted: CV5687.pdf
Successfully extracted: CV5695.pdf
Unsupported file: CV6217.doc
Successfully extracted: CV6425.pdf
Successfully extracted: CV6568.pdf
Unsupported file: CV7309.doc
Successfully extracted: CV7889.pdf
Unsupported file: CV8279.doc
Successfully extracted: CV8299.pdf
Successfully extracted: CV8316.pdf
Successfully extracted: CV8381.docx
Successfully extracted: CV8671.pdf
Unsupported file: CV8875.doc
Unsupported file: CV9308.doc
Successfully extracted: CV9458.pdf
Successfully extracted: CV9497.pdf
Successfully extracted: CV9504.pdf
Unsupported file: CV9534.doc
Succe

Successfully extracted: CV10088.pdf
Successfully extracted: CV10893.pdf
Successfully extracted: CV11249.pdf
Unsupported file: CV11502.doc
Successfully extracted: CV11588.pdf
Successfully extracted: CV12157.pdf
Successfully extracted: CV12241.pdf
Successfully extracted: CV12580.pdf
Successfully extracted: CV12716.pdf
Successfully extracted: CV12768.pdf
Successfully extracted: CV13364.pdf


Successfully extracted: CV14274.pdf
Successfully extracted: CV14503.pdf
Successfully extracted: CV14806.pdf
Successfully extracted: CV15169.pdf
Successfully extracted: CV15193.pdf
Successfully extracted: CV15267.pdf
Successfully extracted: CV15592.pdf
Unsupported file: CV15666.doc
Successfully extracted: CV15833.pdf
Successfully extracted: CV16264.pdf
Successfully extracted: CV16333.pdf
Successfully extracted: CV17591.pdf
Successfully extracted: CV17849.pdf

Total readable CVs: 45


In [ ]:
!apt-get -qq install antiword

Selecting previously unselected package antiword.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../antiword_0.37-16_amd64.deb ...
Unpacking antiword (0.37-16) ...
Setting up antiword (0.37-16) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
import subprocess

for filename in uploaded_files.keys():

    if filename.lower().endswith(".doc"):

        try:
            result = subprocess.run(
                ["antiword", filename],
                capture_output=True,
                text=True
            )

            text = result.stdout

            if text.strip():
                all_cvs.append({
                    "filename": filename,
                    "text": text
                })
                print("Successfully extracted:", filename)
            else:
                print("No readable text:", filename)

        except Exception as e:
            print("Error:", filename, e)

print("\nTOTAL READABLE CVs:", len(all_cvs))

No readable text: CV1917.doc
Successfully extracted: CV3188.doc
Successfully extracted: CV6217.doc
Successfully extracted: CV7309.doc
Successfully extracted: CV8279.doc
Successfully extracted: CV8875.doc
Successfully extracted: CV9308.doc
Successfully extracted: CV9534.doc
Successfully extracted: CV11502.doc
Successfully extracted: CV15666.doc

TOTAL READABLE CVs: 54


In [ ]:
import re

for cv in all_cvs:

    text = cv["text"]

    # Remove extra spaces and blank lines
    text = re.sub(r'\s+', ' ', text)

    # Remove unwanted characters
    text = re.sub(r'[^\x20-\x7E]', ' ', text)

    cv["text"] = text.strip()

print("CV text cleaning completed.")

print("Total CVs:", len(all_cvs))

print("\nExample:")
print(all_cvs[0]["filename"])
print(all_cvs[0]["text"][:500])

CV text cleaning completed.
Total CVs: 54

Example:
CV35.pdf
Assistant Professor Dr. Zyad Shaaban CURRICULUM VITAE (Last Updated: 26-12-2008) ADDRESS Department of Compuer Science, Faculty of Information Technology, Applied Science University, PERSONAL Born: Jan. 24,1969; Place: Kufranjah, Jordan Nationality: Jordanian DATA EDUCATION B. S. Compuer Science, Yarmouk University, Jrbid, Jordan, 1992. Ph. D. Computer Science, University of Technology, Malaysia, 1996. TITLES OF THESES Ph. D. Algorithms for Off-Line Upper-Case Hand-Written Text Recognition and I


In [ ]:
chunks = []

chunk_size = 500
overlap = 100

for cv in all_cvs:
    text = cv["text"]

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunks.append({
            "filename": cv["filename"],
            "text": text[start:end]
        })

        start += chunk_size - overlap

print("Total CVs:", len(all_cvs))
print("Total chunks:", len(chunks))

Total CVs: 54
Total chunks: 1772


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [ ]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding generation completed!")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

Embedding generation completed!
Embedding shape: (1772, 384)


In [ ]:
import faiss
import numpy as np

# Get the embedding size
dimension = embeddings.shape[1]

# Create FAISS index
faiss_index = faiss.IndexFlatIP(dimension)

# Convert embeddings into float32
embedding_vectors = np.array(
    embeddings,
    dtype="float32"
)

# Add all CV embeddings
faiss_index.add(embedding_vectors)

print("FAISS database created successfully!")
print("Total vectors stored:", faiss_index.ntotal)


FAISS database created successfully!
Total vectors stored: 1772


In [ ]:
print("Total CVs:", len(all_cvs))
print("Total chunks:", len(chunks))
print("Total FAISS vectors:", faiss_index.ntotal)

Total CVs: 54
Total chunks: 1772
Total FAISS vectors: 1772


In [ ]:
def retrieve_cvs(question, top_k=10):

    # Convert the question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    # Search FAISS database
    scores, indices = faiss_index.search(
        np.array(question_embedding, dtype="float32"),
        top_k
    )

    retrieved_chunks = []

    for index, score in zip(indices[0], scores[0]):

        if index != -1:

            retrieved_chunks.append({
                "filename": chunks[index]["filename"],
                "text": chunks[index]["text"],
                "score": float(score)
            })

    return retrieved_chunks


# Test the retrieval system
results = retrieve_cvs(
    "Which candidates have Python programming experience?"
)

for result in results:

    print("CV:", result["filename"])
    print("Similarity:", round(result["score"], 3))
    print("Text:", result["text"][:300])
    print("-" * 50)

CV: CV9458.pdf
Similarity: 0.464
Text: introductory courses. Skills: Programming: Unix, Linux Administration, Unicos, X-Windows, Windows 3.1/95/98/2000/NT/XP, Motif, Xlib, C/C++, SML, Objective Caml, Fish, Haskell, Pascal, Prolog, LaTeX, MPI, OpenGL, HTML. Certificates:   Message Passing Programming, Edinburgh Parallel Computing Center, 
--------------------------------------------------
CV: CV10088.pdf
Similarity: 0.454
Text: l Undergraduate Faculty Computer and Information Sciences University Mansoura University Year 2011/2012 (second semester) 6 Course name Programming Language Design Code CS5125 level Postgraduate Faculty Computer and Information Sciences University Mansoura University Year 2011/2012 (second semester)
--------------------------------------------------
CV: CV15193.pdf
Similarity: 0.442
Text: rolog, SML, Unix programming, C/C++, Office Products. Also I gave tutorials to the 3rd year students for the Artificial Intelligence course. - Lab Demonstrator January 1995   Ma

In [ ]:
keyword = "python"

python_candidates = []

for cv in all_cvs:
    if keyword in cv["text"].lower():
        python_candidates.append({
            "filename": cv["filename"],
            "evidence": [
                sentence.strip()
                for sentence in cv["text"].split(".")
                if keyword in sentence.lower()
            ]
        })

print("Candidates mentioning Python:", len(python_candidates))

for candidate in python_candidates:
    print("\nCV:", candidate["filename"])
    for evidence in candidate["evidence"][:3]:
        print("Evidence:", evidence)

Candidates mentioning Python: 2

CV: CV10077.pdf
Evidence: C/C++ , Fortran, and Python Programming

CV: CV16264.pdf
Evidence: 04 LST System Software, Networks   TCP/IP, Sockets, SMTP, POP, HTTP, FTP   IBM MQ Series   IBM DB/2 & IBM Universal Database Java Skills   J2EE -   J2EE - JSP   J2EE - Servlets   J2EE - LDAP   J2EE - EJB   Java - Applications / Swing Web Technologies   JavaScript   HTML   XML / XSL   SAX / DOM   HTTP / HTTPS   Websphere Portal Toolkit   CSS   HTML5 Programming Languages   Java, J2EE, JME   C/C++   CLIPS, Prolog   Perl   Python   AMPL /CPLEX Development Environments   Rational Application Developer / Software Architect 7


In [ ]:
def prepare_context(candidates):
    context = ""

    for candidate in candidates:
        context += f"\nCV: {candidate['filename']}\n"

        for evidence in candidate["evidence"]:
            context += f"Evidence: {evidence}\n"

    return context

context = prepare_context(python_candidates)

print(context)


CV: CV10077.pdf
Evidence: C/C++ , Fortran, and Python Programming

CV: CV16264.pdf
Evidence: 04 LST System Software, Networks   TCP/IP, Sockets, SMTP, POP, HTTP, FTP   IBM MQ Series   IBM DB/2 & IBM Universal Database Java Skills   J2EE -   J2EE - JSP   J2EE - Servlets   J2EE - LDAP   J2EE - EJB   Java - Applications / Swing Web Technologies   JavaScript   HTML   XML / XSL   SAX / DOM   HTTP / HTTPS   Websphere Portal Toolkit   CSS   HTML5 Programming Languages   Java, J2EE, JME   C/C++   CLIPS, Prolog   Perl   Python   AMPL /CPLEX Development Environments   Rational Application Developer / Software Architect 7



In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("AI language model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

AI language model loaded successfully!


In [ ]:
question = "Which candidates have Python experience?"

prompt = f"""
Answer the question using only the CV evidence below.

If a candidate's CV mentions Python, include the CV filename
and the evidence.

Do not invent information.

CV Evidence:
{context}

Question: {question}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = llm.generate(
    **inputs,
    max_new_tokens=120,
    num_beams=4
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("QUESTION:", question)
print("\nAI ANSWER:")
print(answer)

QUESTION: Which candidates have Python experience?

AI ANSWER:
IBM MQ Series IBM DB/2 & IBM Universal Database Java Skills J2EE - J2EE - JSP J2EE - Servlets J2EE - LDAP J2EE - EJB Java - Applications / Swing Web Technologies JavaScript HTML XML / XSL SAX / DOM HTTP / HTTPS Websphere Portal Toolkit CSS HTML5 Programming Languages Java, J2EE, JME C/C++ CLIPS, Prolog Perl Python AMPL /C


In [ ]:
def answer_question(question, candidates):
    if not candidates:
        return "No candidates mentioning the requested skill were found."

    answer = "Candidates mentioning Python:\n\n"

    for candidate in candidates:
        answer += f"CV: {candidate['filename']}\n"

        for evidence in candidate["evidence"][:2]:
            answer += f"Evidence: {evidence}\n"

        answer += "\n"

    return answer


final_answer = answer_question(question, python_candidates)

print("QUESTION:", question)
print("\nFINAL ANSWER:")
print(final_answer)

QUESTION: Which candidates have Python experience?

FINAL ANSWER:
Candidates mentioning Python:

CV: CV10077.pdf
Evidence: C/C++ , Fortran, and Python Programming

CV: CV16264.pdf
Evidence: 04 LST System Software, Networks   TCP/IP, Sockets, SMTP, POP, HTTP, FTP   IBM MQ Series   IBM DB/2 & IBM Universal Database Java Skills   J2EE -   J2EE - JSP   J2EE - Servlets   J2EE - LDAP   J2EE - EJB   Java - Applications / Swing Web Technologies   JavaScript   HTML   XML / XSL   SAX / DOM   HTTP / HTTPS   Websphere Portal Toolkit   CSS   HTML5 Programming Languages   Java, J2EE, JME   C/C++   CLIPS, Prolog   Perl   Python   AMPL /CPLEX Development Environments   Rational Application Developer / Software Architect 7




In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def search_cv(question):
    question = question.strip()

    if not question:
        return "Please enter a question."

    # Find the requested skill in the CV collection
    words = question.lower().split()

    skill = None

    for word in words:
        if word in ["python", "java", "sql", "c++", "tensorflow", "pytorch"]:
            skill = word
            break

    if not skill:
        return "Currently, please ask about Python, Java, SQL, C++, TensorFlow, or PyTorch."

    matching_cvs = []

    for cv in all_cvs:
        if skill in cv["text"].lower():
            matching_cvs.append({
                "filename": cv["filename"],
                "evidence": [
                    sentence.strip()
                    for sentence in cv["text"].split(".")
                    if skill in sentence.lower()
                ]
            })

    if not matching_cvs:
        return f"No CVs mentioning {skill} were found."

    result = f"Candidates mentioning {skill}:\n\n"

    for candidate in matching_cvs:
        result += f"CV: {candidate['filename']}\n"

        for evidence in candidate["evidence"][:2]:
            result += f"Evidence: {evidence}\n"

        result += "\n"

    return result


demo = gr.Interface(
    fn=search_cv,
    inputs=gr.Textbox(
        label="Ask a question about the CVs",
        placeholder="Which candidates have Python experience?"
    ),
    outputs=gr.Textbox(label="Search results", lines=12),
    title="CV Search Assistant",
    description="Search candidate CVs using skills and supporting evidence."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1fa7c6794ade2968d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
